In [ ]:
import base64
import io

from matplotlib import image as mpimg, pyplot as plt

from e2b_code_interpreter import CodeInterpreter

code = """
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 20, 100)
y = np.sin(x)

plt.plot(x, y)
plt.show()
"""

with CodeInterpreter(api_key='e2b_f761407216ad53c65ebb2b59fba2f6fe003c7373') as sandbox:
    # you can install dependencies in "jupyter notebook style"
    # sandbox.notebook.exec_cell("!pip install matplotlib")

    # plot random graph
    execution = sandbox.notebook.exec_cell(code)

# there's your image
image = execution.results[0].png

# example how to show the image / prove it works
i = base64.b64decode(image)
i = io.BytesIO(i)
i = mpimg.imread(i, format='PNG')

plt.imshow(i, interpolation='nearest')
plt.show()


In [22]:
import os
import json

from typing import Any, List
from langchain_core.tools import Tool
from pydantic.v1 import BaseModel, Field
from e2b_code_interpreter import CodeInterpreter
from langchain_core.messages import BaseMessage, ToolMessage
from langchain.agents.output_parsers.tools import (
    ToolAgentAction,
)


class LangchainCodeInterpreterToolInput(BaseModel):
    code: str = Field(description="Python code to execute.")


class CodeInterpreterFunctionTool:
    """
    This class calls arbitrary code against a Python Jupyter notebook.
    It requires an E2B_API_KEY to create a sandbox.
    """

    tool_name: str = "code_interpreter"

    def __init__(self):
        # Instantiate the E2B sandbox - this is a long lived object
        # that's pinging E2B cloud to keep the sandbox alive.
        if "E2B_API_KEY" not in os.environ:
            raise Exception(
                "Code Interpreter tool called while E2B_API_KEY environment variable is not set. Please get your E2B api key here https://e2b.dev/docs and set the E2B_API_KEY environment variable."
            )
        self.code_interpreter = CodeInterpreter()

    def call(self, parameters: dict, **kwargs: Any):
        code = parameters.get("code", "")
        print(f"***Code Interpreting...\n{code}\n====")
        execution = self.code_interpreter.notebook.exec_cell(code)
        return {
            "results": execution.results,
            "stdout": execution.logs.stdout,
            "stderr": execution.logs.stderr,
            "error": execution.error,
        }

    def close(self):
        self.code_interpreter.close()

    # langchain does not return a dict as a parameter, only a code string
    def langchain_call(self, code: str):
        return self.call({"code": code})

    def to_langchain_tool(self) -> Tool:
        tool = Tool(
            name=self.tool_name,
            description="Execute python code in a Jupyter notebook cell and returns any rich data (eg charts), stdout, stderr, and error.",
            func=self.langchain_call,
        )
        tool.args_schema = LangchainCodeInterpreterToolInput
        return tool

    @staticmethod
    def format_to_tool_message(
        agent_action: ToolAgentAction,
        observation: dict,
    ) -> List[BaseMessage]:
        """
        Format the output of the CodeInterpreter tool to be returned as a ToolMessage.
        """
        new_messages = list(agent_action.message_log)

        # TODO: Add info about the results for the LLM
        content = json.dumps(
            {k: v for k, v in observation.items() if k not in ("results")}, indent=2
        )
        new_messages.append(
            ToolMessage(content=content, tool_call_id=agent_action.tool_call_id)
        )

        return new_messages

In [23]:
from typing import List, Sequence, Tuple
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnablePassthrough
from langchain.agents.output_parsers.tools import (
    ToolAgentAction,
    ToolsAgentOutputParser,
)
from jqcode.c0_init_env_models import get_langchain_chat_model

def format_to_tool_messages(
    intermediate_steps: Sequence[Tuple[ToolAgentAction, dict]],
) -> List[BaseMessage]:
    messages = []
    for agent_action, observation in intermediate_steps:
        if agent_action.tool == CodeInterpreterFunctionTool.tool_name:
            new_messages = CodeInterpreterFunctionTool.format_to_tool_message(
                agent_action,
                observation,
            )
            print(new_messages)
            messages.extend([new for new in new_messages if new not in messages])
        else:
            # Handle other tools
            print("Not handling tool: ", agent_action.tool)

    return messages



In [25]:
# 1. Pick your favorite llm
llm = get_langchain_chat_model()

# 2. Initialize the code interpreter tool
code_interpreter = CodeInterpreterFunctionTool()
code_interpreter_tool = code_interpreter.to_langchain_tool()
tools = [code_interpreter_tool]

llm_with_tools = llm.bind_tools(tools)

05/08/2024 20:36:10 - [INFO] -e2b.sandbox.main->>>    Creating sandbox code-interpreter-stateful
05/08/2024 20:36:10 - [INFO] -e2b.sandbox.sandbox_connection->>>    Sandbox for template code-interpreter-stateful initialized


当前使用的模型为:Langchain ChatOpenAI, qwen1.5-chat, API_URL为http://10.1.150.105:9997/v1


05/08/2024 20:36:11 - [INFO] -e2b.sandbox.main->>>    Opening sandbox code-interpreter-stateful
05/08/2024 20:36:15 - [INFO] -e2b.sandbox.sandbox_connection->>>    Sandbox 1tsfj5yvigmgc5gmgqz2 created (id:iqlxltnytr9lmzowcn8is-749fb54d)
05/08/2024 20:36:16 - [INFO] -e2b.sandbox.sandbox_connection->>>    Started refreshing sandbox 1tsfj5yvigmgc5gmgqz2 (id: iqlxltnytr9lmzowcn8is-749fb54d)
05/08/2024 20:36:16 - [INFO] -e2b.sandbox.sandbox_rpc->>>    WebSocket waiting to start
05/08/2024 20:36:24 - [INFO] -e2b.sandbox.websocket_client->>>    WebSocket connected to wss://49982-iqlxltnytr9lmzowcn8is-749fb54d.e2b.dev/ws
05/08/2024 20:36:24 - [INFO] -e2b.sandbox.sandbox_rpc->>>    WebSocket started
05/08/2024 20:36:25 - [INFO] -e2b.sandbox.main->>>    Sandbox code-interpreter-stateful opened


05/08/2024 20:36:35 - [INFO] -e2b.sandbox.websocket_client->>>    WebSocket connected to wss://8888-iqlxltnytr9lmzowcn8is-749fb54d.e2b.dev/api/kernels/ce3348c5-98cd-4945-b921-7102146eefdb/channels
05/08/2024 20:38:11 - [INFO] -e2b.sandbox.websocket_client->>>    WebSocket stopped
05/08/2024 20:38:14 - [INFO] -e2b.sandbox.sandbox_connection->>>    Stopped refreshing sandbox (id: iqlxltnytr9lmzowcn8is-749fb54d)


In [26]:
llm_with_tools

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7f1a78393150>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7f1a78391610>, model_name='qwen1.5-chat', temperature=0.2, openai_api_key=SecretStr('**********'), openai_api_base='http://10.1.150.105:9997/v1', openai_proxy='', streaming=True, max_tokens=5000), kwargs={'tools': [{'type': 'function', 'function': {'name': 'code_interpreter', 'description': 'Execute python code in a Jupyter notebook cell and returns any rich data (eg charts), stdout, stderr, and error.', 'parameters': {'type': 'object', 'properties': {'code': {'description': 'Python code to execute.', 'type': 'string'}}, 'required': ['code']}}}]})

In [28]:
# 3. Define the prompt
prompt = ChatPromptTemplate.from_messages(
    [("human", "{input}"), ("placeholder", "{agent_scratchpad}")]
)

# 4. Define the agent
agent = (
    RunnablePassthrough.assign(
        agent_scratchpad=lambda x: format_to_tool_messages(x["intermediate_steps"])
    )
    | prompt
    | llm_with_tools
    | ToolsAgentOutputParser()
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
)

# 5. Invoke the agent
result = agent_executor.invoke({"input": "请生成可用的python代码，画出正弦曲线图"})
code_interpreter.close()



> Entering new AgentExecutor chain...


05/08/2024 20:40:32 - [INFO] -httpx->>>    HTTP Request: POST http://10.1.150.105:9997/v1/chat/completions "HTTP/1.1 200 OK"





> Finished chain.


05/08/2024 20:40:37 - [INFO] -e2b.sandbox.sandbox_connection->>>    Sandbox closed


In [ ]:
# Each intermediate step is a Tuple[ToolAgentAction, dict]
result

In [ ]:
result["intermediate_steps"][0][1]["results"][0]